In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abhavyajha/amazon-ml/utils/validate_submission.py
/kaggle/input/datasets/abhavyajha/amazon-ml/test/test_source2.tsv
/kaggle/input/datasets/abhavyajha/amazon-ml/test/test_source3.tsv
/kaggle/input/datasets/abhavyajha/amazon-ml/test/test_source1.tsv
/kaggle/input/datasets/abhavyajha/amazon-ml/train/train_ground_truth.tsv
/kaggle/input/datasets/abhavyajha/amazon-ml/train/train_source3.tsv
/kaggle/input/datasets/abhavyajha/amazon-ml/train/train_source2.tsv
/kaggle/input/datasets/abhavyajha/amazon-ml/train/train_source1.tsv


In [2]:
!git clone -b claude/entity-matching-ranking-3a3p0e https://github.com/ajha2005/amazon-ml.git /kaggle/working/amazon-ml
!git -C /kaggle/working/amazon-ml checkout -q 318e742
!pip install -q rapidfuzz

import os, glob
for d in ('train', 'test'):
    os.makedirs(f'/kaggle/working/amazon-ml/dataset/{d}', exist_ok=True)
for f in glob.glob('/kaggle/input/**/*.tsv', recursive=True):
    dst = 'train' if '/train/' in f else 'test'
    target = f'/kaggle/working/amazon-ml/dataset/{dst}/{os.path.basename(f)}'
    if not os.path.exists(target):
        os.symlink(f, target)
print('TRAIN:', sorted(os.listdir('/kaggle/working/amazon-ml/dataset/train')))
print('TEST :', sorted(os.listdir('/kaggle/working/amazon-ml/dataset/test')))
!git -C /kaggle/working/amazon-ml log --oneline -1

Cloning into '/kaggle/working/amazon-ml'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 136 (delta 73), reused 106 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 65.57 KiB | 2.26 MiB/s, done.
Resolving deltas: 100% (73/73), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.9 MB/s eta 0:00:00
TRAIN: ['train_ground_truth.tsv', 'train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv']
TEST : ['test_source1.tsv', 'test_source2.tsv', 'test_source3.tsv']
318e742 (HEAD, origin/claude/entity-matching-ranking-3a3p0e, claude/entity-matching-ranking-3a3p0e) Normalize French, Indian and US spelling variants


In [3]:
!python -u /kaggle/working/amazon-ml/code/business_entity_resolution/src/run_pipeline.py --mode both


[1/4] Loading + normalizing train data...
  train_source1.tsv: 2,206,821 rows
  train_source2.tsv: 5,034,616 rows
  train_source3.tsv: 5,285,603 rows
    Normalized 3,000,000/10,320,219 rows...
    Normalized 5,500,000/10,320,219 rows...
    Normalized 8,000,000/10,320,219 rows...
    Normalized 10,320,219/10,320,219 rows...
  Normalized in 780s
  Cached -> train_norm_n3.pkl (36s)
  S1=2,206,821  S2+S3=10,320,219

[2/4] Blocking (top-10 per S1 + top-3 per S2/S3, df_cap=2000)...
  Countries in S1: ['us', 'india'] | in S2/S3: ['india', 'us']
  [us] S1=1,323,633  S2/S3=6,186,873
    vocab 9,772,269 tokens, kept 9,764,307 with df <= 2000
    S1-side 13,235,902 + S2/S3-side 18,496,997 -> 23,220,092 unique pairs
    -> 23,220,092 pairs in 812s
  [india] S1=883,188  S2/S3=4,133,346
    vocab 6,251,684 tokens, kept 6,244,362 with df <= 2000
    S1-side 8,831,400 + S2/S3-side 12,327,681 -> 15,486,386 unique pairs
    -> 15,486,386 pairs in 720s
  Total candidate pairs: 38,706,478 (17.5 per S1 

In [4]:
!cp /kaggle/working/amazon-ml/output/*.tsv /kaggle/working/
!cp /kaggle/working/amazon-ml/output/selection_params.json /kaggle/working/
!ls -lah /kaggle/working/*.tsv

-rw-r--r-- 1 root root 118M Sep 27 06:54 /kaggle/working/candidate_pairs.tsv
-rw-r--r-- 1 root root  90M Sep 27 06:54 /kaggle/working/matching_results.tsv
